# Week 29, Day 5: Encapsulation in Python

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w029d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Team:
    def __init__(self):
        self._players = []

    def add(self, name):
        if len(self._players) >= 3:
            raise ValueError("team is full")
        self._players.append(name)

    @property
    def players(self):
        return self._players

t = Team()
for n in ["Ada", "Bo", "Cy"]:
    t.add(n)
t.players.append("Di")
t.players.append("Ed")
print(t.players)

**Predict before you run:** what will this print?



In [ ]:
class Account:
    def __init__(self):
        self.__pin = "1234"

    def check(self, pin):
        return pin == self.__pin

a = Account()
print(a.check("1234"))
try:
    print(a.__pin)
except AttributeError as e:
    print("AttributeError")
print(vars(a), a._Account__pin)

In [ ]:
class Team:
    MAX = 3

    def __init__(self, players=()):
        self._players = []
        for p in players:
            self.add(p)

    def add(self, name):
        if len(self._players) >= Team.MAX:
            raise ValueError("team is full")
        if name in self._players:
            raise ValueError(f"{name} is already in the team")
        self._players.append(name)

    @property
    def players(self):
        return tuple(self._players)

    def has(self, name):
        return name in self._players

t = Team(["Ada", "Bo"])
view = t.players
try:
    view.append("Cy")
except AttributeError:
    print("can't change the view")
print(t.players, t.has("Bo"), len(t.players))

In [ ]:
class Temperature:
    """Version 2: stores kelvin inside. The interface is the same as version 1, which stored celsius."""

    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return round(self._kelvin - 273.15, 2)

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._kelvin = value + 273.15

    @property
    def fahrenheit(self):
        return round(self.celsius * 9 / 5 + 32, 2)

t = Temperature(21.5)
t.celsius += 1
print(t.celsius, t.fahrenheit)

In [ ]:
class Point:
    def __init__(self, x, y):
        self._x, self._y = x, y

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y

    def moved(self, dx, dy):
        return Point(self._x + dx, self._y + dy)

    def __repr__(self):
        return f"Point({self._x!r}, {self._y!r})"

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self._x, self._y) == (other._x, other._y)

    def __hash__(self):
        return hash((self._x, self._y))

start = Point(0, 0)
end = start.moved(3, 4)
print(start, end, {start, end, Point(3, 4)})

> **Key idea.** Public names are the interface; a leading underscore means internal; two underscores add name mangling, which prevents clashes in subclasses but isn't security. Don't let internal mutable data leak: return copies or tuples, copy what you're given, or offer queries. Keep the interface small, so the implementation can change freely. Immutable value classes never change, so they're safe to share and hash.

### Exercise: No leaks

Write `Playlist(name, songs=())` keeping songs in an internal list (copied from `songs`). `add(song)` refuses duplicates (`ValueError`), `remove(song)` raises `KeyError` if it isn't there, the read-only property `songs` returns a **tuple**, and `size()` returns the number of songs. No outside code should be able to change the playlist except through `add` and `remove`.

In [ ]:
class Playlist:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"no leaks\"\n    P = ns[\"Playlist\"]\n    given = [\"a\", \"b\"]\n    p = P(\"Mix\", given)\n    given.append(\"c\")\n    assert p.songs == (\"a\", \"b\") and p.size() == 2\n    p.add(\"c\")\n    try:\n        p.songs.append(\"d\")\n        assert False, \"songs should be a tuple\"\n    except AttributeError:\n        pass\n    try:\n        p.songs = ()\n        assert False, \"songs should be read-only\"\n    except AttributeError:\n        pass\n    p.remove(\"a\")\n    assert p.songs == (\"b\", \"c\")\n    for f, err in [(lambda: p.add(\"b\"), ValueError), (lambda: p.remove(\"zz\"), KeyError)]:\n        try:\n            f()\n            assert False, f\"should raise {err.__name__}\"\n        except err:\n            pass")

### Exercise: Same interface, new inside

`Wallet` version 1 stores a float of pounds, which drifts (`0.1 + 0.2`). Write version 2 that stores **whole pence** in `_pence` but keeps exactly the same interface: `Wallet(pounds)`, `add(pounds)`, `spend(pounds)` (raising `ValueError` if there isn't enough), and the property `pounds` (a float, rounded to 2 places). The checks use only that interface.

In [ ]:
class Wallet:
    def __init__(self, pounds=0.0):
        self.pounds = pounds

    def add(self, pounds):
        self.pounds += pounds

    def spend(self, pounds):
        if pounds > self.pounds:
            raise ValueError("not enough")
        self.pounds -= pounds

w = Wallet()
for _ in range(10):
    w.add(0.1)
w.spend(0.3)
print(w.pounds)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"exact\"\n    expect_output(\"0.7\")\n    W = ns[\"Wallet\"]\n    w = W(0.3)\n    w.spend(0.1); w.spend(0.2)\n    assert w.pounds == 0.0\n    try:\n        w.spend(0.01)\n        assert False, \"spending from nothing should raise ValueError\"\n    except ValueError:\n        pass\n    assert \"_pence\" in vars(W(1.25)) and vars(W(1.25))[\"_pence\"] == 125")

### Exercise: An immutable date range

Write `DateRange(start, end)` for two `date`s (start not after end, or `ValueError`), immutable: read-only properties `start`, `end` and `days` (inclusive count), and methods that return **new** ranges: `extended(n)` (end moved n days later) and `shifted(n)` (both moved). Also `contains(d)`, `overlaps(other)`, `__repr__` like `DateRange(2026-11-01, 2026-11-07)`, `__eq__` and `__hash__`.

In [ ]:
from datetime import date, timedelta

class DateRange:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from datetime import date as _d\ndef test_values():\n    \"values\"\n    R = ns[\"DateRange\"]\n    r = R(_d(2026, 11, 1), _d(2026, 11, 7))\n    assert r.days == 7 and repr(r) == \"DateRange(2026-11-01, 2026-11-07)\"\n    e, s = r.extended(3), r.shifted(-1)\n    assert (e.end, s.start, s.end) == (_d(2026, 11, 10), _d(2026, 10, 31), _d(2026, 11, 6))\n    assert r.end == _d(2026, 11, 7), \"r must not change\"\n    assert r.contains(_d(2026, 11, 7)) and not r.contains(_d(2026, 11, 8))\n    assert r.overlaps(R(_d(2026, 11, 7), _d(2026, 12, 1))) and not r.overlaps(R(_d(2026, 11, 8), _d(2026, 11, 9)))\n    assert len({r, R(_d(2026, 11, 1), _d(2026, 11, 7)), e}) == 2\ndef test_immutable():\n    \"immutable and checked\"\n    R = ns[\"DateRange\"]\n    r = R(_d(2026, 1, 1), _d(2026, 1, 1))\n    for name in (\"start\", \"end\", \"days\"):\n        try:\n            setattr(r, name, None)\n            assert False, f\"{name} should be read-only\"\n        except AttributeError:\n            pass\n    try:\n        R(_d(2026, 1, 2), _d(2026, 1, 1))\n        assert False, \"start after end should raise ValueError\"\n    except ValueError:\n        pass")

### Exercise: Predict the names



In [ ]:
class Safe:
    def __init__(self):
        self.label = "home"
        self._code = 12
        self.__key = "k"
        self.__dunder__ = "d"

s = Safe()
p = ___   # sorted(vars(s))
q = ___   # hasattr(s, "__key")
r = ___   # s._Safe__key

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"_Safe__key\", \"__dunder__\", \"_code\", \"label\"] and q is False and r == \"k\"")

### Bug hunt: The leaky inventory

`Inventory.stock` lets callers change the stock directly, skipping the "never negative" rule: the last line sets it to -5. Fix the property so outside code can only read a copy.

In [ ]:
class Inventory:
    def __init__(self):
        self._stock = {}

    def add(self, item, n):
        if n <= 0:
            raise ValueError("n must be positive")
        self._stock[item] = self._stock.get(item, 0) + n

    @property
    def stock(self):
        return self._stock

inv = Inventory()
inv.add("pens", 10)
inv.stock["pens"] = -5
print(inv.stock)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"unchanged\"\n    expect_output(\"{'pens': 10}\")")

### Bug hunt: Mangled out of reach

The program wants to print the score, but `p.__score` raises `AttributeError`. Don't reach into `_Player__score`: add a read-only property `score` to the class, and use it.

In [ ]:
class Player:
    def __init__(self, name):
        self.name = name
        self.__score = 0

    def win(self):
        self.__score += 3

p = Player("Ada")
p.win(); p.win()
print(p.__score)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the score\"\n    expect_output(\"6\")\n    assert \"_Player__score\" not in source")

### Puzzle: A safe view

Arrange the lines into a class that keeps tags private and shows them only as a sorted tuple. One line is a trap.

(Put these lines in order on the website.)

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an account built from its history

Write `Account(owner)` that stores **only** its owner and a private list of transactions, each an immutable tuple `(kind, pence)` where kind is `"deposit"` or `"withdrawal"`. The balance isn't stored; the read-only property `balance` works it out from the history. Provide `deposit(pence)` and `withdraw(pence)` (whole positive numbers; no overdrafts: `ValueError`), a read-only property `history` returning a **tuple** of the transactions, and `statement()` returning lines like `deposit     £12.50   balance £12.50`: the kind in 10 characters, the amount in 8 (right-aligned), three spaces, then `balance` and the running balance. The checks will try to break the account through its public names; none of their attempts should succeed.

In [ ]:
class Account:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"balance and statement\"\n    A = ns[\"Account\"]\n    a = A(\"Ada\")\n    a.deposit(1250); a.deposit(500); a.withdraw(300)\n    assert a.balance == 1450 and a.history == ((\"deposit\", 1250), (\"deposit\", 500), (\"withdrawal\", 300))\n    assert a.statement() == [\"deposit     \u00a312.50   balance \u00a312.50\", \"deposit      \u00a35.00   balance \u00a317.50\", \"withdrawal   \u00a33.00   balance \u00a314.50\"]\n    allowed = {\"owner\", \"_owner\", \"_history\", \"_transactions\", \"_Account__history\", \"_Account__transactions\"}\n    assert len(vars(a)) == 2 and set(vars(a)) <= allowed, \"store only the owner and the history\"\ndef test_tamper():\n    \"tampering fails\"\n    A = ns[\"Account\"]\n    a = A(\"Ada\")\n    a.deposit(100)\n    for f, err in [(lambda: setattr(a, \"balance\", 10**6), AttributeError), (lambda: setattr(a, \"history\", ()), AttributeError),\n                   (lambda: a.history.append((\"deposit\", 5)), AttributeError), (lambda: a.withdraw(101), ValueError),\n                   (lambda: a.deposit(0), ValueError), (lambda: a.deposit(2.5), ValueError)]:\n        try:\n            f()\n            assert False, \"should be refused\"\n        except err:\n            pass\n    assert a.balance == 100 and len(a.history) == 1")

## Warm-up

`players` is a read-only property, and `add` checks the limit. But the property hands out the team's **own** list, and whoever gets it can change it however they like: the team now has five players. Encapsulation is about more than names.

## Three kinds of name

Python has no `private` keyword. Instead, names follow a convention every Python programmer knows:

| Name | Meaning |
|---|---|
| `balance` | **public**: part of the class's interface; use it freely |
| `_balance` | **internal**: an implementation detail; code outside the class shouldn't touch it, and it may change without warning |
| `__balance` | **name-mangled**: like internal, plus protection against clashes in subclasses (next section) |

Nothing stops you reading `acc._balance`, and occasionally (in tests, or when debugging) you might. The underscore is a promise from the class's author: "everything without an underscore will keep working; anything with one, I may change". This is often summed up as "we're all consenting adults here": Python trusts programmers to respect the convention rather than locking things away.

The same convention works for modules: `from shop import *` skips names starting with `_`, and linters warn when you use another module's internal names.

## Name mangling

A name starting with **two** underscores (and not ending with two, like `__init__`) inside a class body is rewritten by Python: `__balance` in class `Account` becomes `_Account__balance`.

So `a.__pin` fails from outside, but `a._Account__pin` works: mangling isn't security. Its real purpose is to stop **subclasses** (Week 30) accidentally overwriting a parent class's internal attributes by using the same name. If a parent class uses `self.__count` and a child class also uses `self.__count`, they become `_Parent__count` and `_Child__count` and don't collide.

In practice, most Python code uses a **single** underscore for internal names, and keeps double underscores for the rare case where a class is designed to be subclassed by people who won't know its internals.

Never store real secrets (passwords, keys) as attributes thinking an underscore hides them: anything in a program's memory can be read by the program.

## Leaks

The warm-up showed the commonest encapsulation bug: returning an internal **mutable** object. Whoever receives it holds a reference to the class's own data, and can change it without going through any of the class's checks.

Three ways to stop a leak:

1. **Return a copy**: `return list(self._players)`. The caller can change their copy freely.
2. **Return an immutable view**: `return tuple(self._players)`, or `frozenset(...)` for a set. Attempts to change it fail loudly.
3. **Don't return the collection at all**: offer queries instead (`count()`, `has(name)`, `__iter__` in Week 31).

The same applies on the way **in**: copy lists given to `__init__` or to setters (Day 3 of last week), so the caller can't change your data afterwards through their reference.

Copying costs a little time and memory; for most classes it's nothing. For very large collections, prefer the third option.

## Interface and implementation

A class's **interface** is what code outside uses: its public methods, properties and attributes. Its **implementation** is how it does it: the internal attributes and helper methods. Keeping the interface small and the implementation hidden gives you the freedom to change the inside later, to fix a bug, speed it up or store data differently, without changing a single line of code that uses the class.

Every program that used version 1 works unchanged with version 2, because none of them ever touched `_celsius`. If any had, the change would have broken it. That's the practical reason for the underscore convention.

**Immutable value classes** take this furthest: no setters at all, so once made, an object never changes. "Changing" one makes a new one, the way string methods do. That makes them safe to share, and to hash (Day 2):

(Week 32's frozen dataclasses write all of that for you.)

## Practice

## Quiz

## Daily challenge

## Before you go

Check the classes you've written for leaks: does any property or method hand out an internal list, dictionary or set? Fix each one. Tomorrow's Forge puts the whole week together: a value class with operators, equality and properties, and a collection class that keeps its data safe.